# RMShell Solve and Postprocess Tutorial

This notebook walks through the current `RMShellModel` API in a little more detail than the examples.

It covers:

1. Building a small shell problem.
2. Creating canonical material and load input groups.
3. Solving for the shell displacement state.
4. Using the postprocessor in a few different ways.
5. Requesting custom postprocessing outputs from forms.
6. Reusing the postprocessor with an externally supplied displacement field.

In [ ]:
import numpy as np
import csdl_alpha as csdl
import dolfinx
import dolfinx.io
import meshio
import ufl
from mpi4py import MPI

from femo_alpha.rm_shell.rm_shell_model import RMShellModel

recorder = csdl.Recorder(inline=True)
recorder.start()

## Build a small shell model

The example below creates a simple cantilevered quadrilateral shell mesh directly in the notebook. For larger problems you would normally load the mesh from your own geometry/meshing pipeline.

In [ ]:
def clamped_boundary(x):
    return np.less(x[0], 1.0e-12)

nx, ny = 4, 2
xs = np.linspace(0.0, 10.0, nx + 1)
ys = np.linspace(0.0, 2.0, ny + 1)
points = np.array([[x, y, 0.0] for y in ys for x in xs], dtype=float)
cells = []
for j in range(ny):
    for i in range(nx):
        n0 = j * (nx + 1) + i
        n1 = n0 + 1
        n3 = n0 + (nx + 1)
        n2 = n3 + 1
        cells.append([n0, n1, n2, n3])

mesh_path = "./rmshell_tutorial_mesh.xdmf"
meshio.write(mesh_path, meshio.Mesh(points, [("quad", np.array(cells, dtype=np.int64))]))
with dolfinx.io.XDMFFile(MPI.COMM_WORLD, mesh_path, "r") as xdmf:
    mesh = xdmf.read_mesh(name="Grid")

shell = RMShellModel(
    mesh,
    shell_bc_func=clamped_boundary,
    element_wise_material=False,
    solve_direct=True,
    record=False,
)

## Build canonical material and load inputs

`RMShellModel` now uses explicit input factories. The input groups returned by these helpers are the objects passed into `solve(...)` and later reused by the postprocessor.

In [ ]:
nn = shell.nn
thickness = csdl.Variable(value=0.1 * np.ones(nn), name="thickness")
E = csdl.Variable(value=1.0e8 * np.ones(nn), name="E")
nu = csdl.Variable(value=0.3 * np.ones(nn), name="nu")
density = csdl.Variable(value=10.0 * np.ones(nn), name="density")

nodal_pressure = csdl.Variable(value=np.zeros((nn, 3)), name="nodal_pressure")
nodal_pressure.value[:, 2] = 5.0
node_disp = csdl.Variable(value=np.zeros((nn, 3)), name="node_disp")

material = shell.material_inputs.from_isotropic(
    E=E,
    nu=nu,
    thickness=thickness,
    density=density,
)
loads = shell.load_inputs.from_fields(
    nodal_pressure=nodal_pressure,
)

## Solve the shell problem

The solver returns a `ShellState`, which packages the input groups, mesh deformation, and solved displacement variables. That state can be handed directly to the postprocessor.

In [ ]:

state = shell.solve(material=material, loads=loads, node_disp=node_disp)

## Default postprocessing bundle

Request the outputs you want, then call `shell.post.compute(...)` once. `shell.post.evaluate(...)` remains available as a compatibility shortcut for the default bundle.

In [ ]:
default_outputs = shell.post.clear().add_default_outputs().compute(state=state)

print("Compliance:", default_outputs.compliance.value)
print("Mass:", default_outputs.mass.value)
print("CG:", default_outputs.cg.value)
print("Tip deflection:", np.max(default_outputs.disp_extracted.value[:, 2]))

## Compute only the outputs you need

For larger workflows, request the postprocessing outputs explicitly. The requested set should contain only the outputs this workflow will compute.

A `ShellPostContext` lets the postprocessor share prepared inputs and cached intermediate results during the compute call.

In [ ]:
post_context = shell.post.context(state=state)

selected_outputs = (
    shell.post.clear()
    .compliance()
    .mass_properties()
    .kinematics()
    .compute(context=post_context)
)

print("Compliance only:", selected_outputs.compliance.value)
print("Mass:", selected_outputs.mass.value)
print("CG:", selected_outputs.cg.value)
print("Displacement field shape:", selected_outputs.disp_extracted.shape)
print("Rotation field shape:", selected_outputs.rotations.shape)

The grouped helpers add related outputs to the requested set. They are useful when outputs naturally belong together, for example mass properties or strain quantities.

In [ ]:
grouped_outputs = shell.post.clear().mass_properties().strains().compute(state=state)

print("Mass helper mass:", grouped_outputs.mass.value)
print("Mass helper cg:", grouped_outputs.cg.value)
print("Mid-strain field shape:", grouped_outputs.mid_strain.shape)
print("Curvature field shape:", grouped_outputs.curvature.shape)

## Register a custom postprocessing output

Custom outputs are requested on `shell.post`. If the new quantity comes from a common form pattern, use a dedicated helper.

Below, `avg_eps_x` computes the area-averaged mid-surface strain component $\varepsilon_{xx}$ over the whole shell. The helper handles the numerator and area normalization internally.

In [ ]:
shell.post.clear().average_strain(
    "avg_eps_x",
    strain_type="mid",
    component="xx",
)

avg_outputs = shell.post.compute(context=post_context)
print("Average epsilon_xx:", avg_outputs.avg_eps_x.value)

You can also register alternative versions of built-in ideas. Here is a second p-norm stress output with different aggregation parameters.

In [ ]:
shell.post.pnorm_stress_custom(
    "pnorm_stress_soft",
    rho=20,
    m=1.0e-6,
)

custom_scalars = shell.post.pnorm_stress().compute(context=post_context)

print("Built-in pnorm_stress:", custom_scalars.pnorm_stress.value)
print("Custom soft pnorm_stress:", custom_scalars.pnorm_stress_soft.value)
print("Average epsilon_xx:", custom_scalars.avg_eps_x.value)

Sometimes you want a quantity that is still best expressed as a form, but does not match any stock helper. In that case, you can register the forms directly.

The example below defines a new scalar output, `mean_w`, equal to the area-average of the transverse displacement field over the shell mid-surface:

$$
\mathrm{mean\_w} = \frac{\int_{\Omega} u_z\,dx}{\int_{\Omega} 1\,dx}.
$$

This is a true custom postprocessing form: it is assembled by the postprocessor, can be reused with an externally supplied displacement, and does not rely on one of the stock helpers.

In [ ]:
def mean_w_numerator(context):
    return context.post_fea.inputs_dict["disp_solid"]["function"][2] * context.region_measure()

shell.post.clear().add_form_ratio(
    "mean_w",
    mean_w_numerator,
    ["disp_solid"],
    lambda context: context.area_form(),
    ["uhat"],
)

mean_w_outputs = shell.post.compute(context=post_context)
print("Area-averaged transverse displacement:", mean_w_outputs.mean_w.value)

## Reuse the postprocessor with an external displacement

Postprocessing is separate from solving. If you already have a displacement field from another source, you can evaluate the same outputs by providing `material`, `loads`, and `displacement` directly.

In [ ]:
external_outputs = shell.post.clear().add_default_outputs().compute(
    material=material,
    loads=loads,
    displacement=state.disp_solid,
    node_disp=node_disp,
)

print("External compliance:", external_outputs.compliance.value)
print("External tip deflection:", np.max(external_outputs.disp_extracted.value[:, 2]))

## Direct generalized load vectors

If an upstream transfer already provides the generalized shell load vector, you can skip field reconstruction entirely:

```python
load_vector = shell.assemble_generalized_load_vector(
    nodal_pressure=nodal_pressure.value,
    node_disp=node_disp.value,
)
loads_vector = shell.load_inputs.from_vector(
    load_vector=load_vector,
)
state_vector = shell.solve(material=material, loads=loads_vector, node_disp=node_disp)
outputs_vector = shell.post.clear().add_default_outputs().compute(state=state_vector)
```

That is the path you want when an external load-transfer operation already computes the generalized right-hand side in shell ordering.